In [1]:
import json
import os
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import pybigtools
from tqdm.auto import tqdm

/home/a/asmith/project_milne_group/software/Regulonado/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Get the paths from my pre-defined TSS scores

In [2]:
tss_scores = pd.read_parquet(
    "/project/milne_group/asmith/Projects/2025-07-19-myeloid-specific-enhancer-identification/data/processed/wimm_atlas/2025-11-26-tss-enrichment-scores-all.parquet"
)
bw_paths_original = tss_scores['path']

In [7]:
bw_paths_original_df = pd.DataFrame({'path': bw_paths_original}).assign(
    name=lambda df: df['path'].apply(lambda x: Path(x).stem)
)
bw_paths_original_df.loc[lambda df: df['name'].str.contains('ChIP')]

,path,name
1109,/project/milne_group/cchahrou/data/2024-10-14_...,ChIP-26754-1_H3K27ac
1112,/project/milne_group/cchahrou/data/2024-10-14_...,ChIP-23003-1_MLL-N
1127,/project/milne_group/cchahrou/data/2024-10-14_...,ChIP-26754-1_MLL-N
1130,/project/milne_group/cchahrou/data/2024-10-14_...,ChIP-26754-1_AF4-C


## Add to the paths

### General datashare glob

In [8]:
datashare_asmith = Path('/ceph/project/milne_group/datashare/asmith/atacseq/')
datashare_asmith

PosixPath('/ceph/project/milne_group/datashare/asmith/atacseq')

In [20]:
exclude = {'capcruncher', 'mcc', 'capture', 'tet', 'hg19', 'mm10', 'mm39', 'input', 'spikein', 'igg', 'homer'}

files = [
    p for p in datashare_asmith.rglob('*.bigWig')
    if not any(ex in str(p).lower() for ex in exclude)
]

bw_paths_datashare_df = pd.DataFrame({'path': files}).assign(
    name=lambda df: df['path'].apply(lambda x: Path(x).stem)
)

## Look in Catherine's nice processed data

In [37]:
cc_cat = Path('/ceph/project/milne_group/cchahrou/data/2025-05-21_cat_all/seqnado_output/bigwigs/deeptools/unscaled')
files_cc_cat = list(cc_cat.rglob('*.bigWig'))
files_cc_cat = [f for f in files_cc_cat if not any(ex in str(f).lower() for ex in exclude)]


cc_sem_chip = Path('/ceph/project/milne_group/cchahrou/data/2025-03-24_chip_sem_archive/seqnado_output/bigwigs/deeptools/unscaled')
files_cc_sem_chip = list(cc_sem_chip.rglob('*.bigWig'))
files_cc_sem_chip = [f for f in files_cc_sem_chip if not any(ex in str(f).lower() for ex in exclude)]

cc_rch_cat = Path('/ceph/project/milne_group/cchahrou/data/2025-06-30_cat_rchacv/seqnado_output/bigwigs/deeptools/unscaled')
files_cc_rch_cat = list(cc_rch_cat.rglob('*.bigWig'))
files_cc_rch_cat = [f for f in files_cc_rch_cat if not any(ex in str(f).lower() for ex in exclude)]

cc_thp1_chip = Path('/ceph/project/milne_group/cchahrou/data/2025-07-21_chip_thp1_archive/seqnado_output/bigwigs/deeptools/unscaled')
files_cc_thp1_chip = list(cc_thp1_chip.rglob('*.bigWig'))
files_cc_thp1_chip = [f for f in files_cc_thp1_chip if not any(ex in str(f).lower() for ex in exclude)]

cc_rna = Path('/ceph/project/milne_group/cchahrou/data/2025-03-16_rna_archive/seqnado_output/bigwigs/deeptools/unscaled')
files_cc_rna = list(cc_rna.rglob('*.bigWig'))
files_cc_rna = [f for f in files_cc_rna if not any(ex in str(f).lower() for ex in exclude)]

### Combine

In [38]:
files_all = (files + 
             files_cc_cat + 
             files_cc_sem_chip + 
             files_cc_rch_cat + 
             files_cc_thp1_chip + 
             files_cc_rna)

bw_paths_all_df = pd.DataFrame({'path': files_all}).assign(
    name=lambda df: df['path'].apply(lambda x: Path(x).stem)
)
bw_paths_all_df.assign(is_duplicate=lambda df: df['name'].duplicated(keep=False))

,path,name,is_duplicate
0,/ceph/project/milne_group/datashare/asmith/ata...,CB9-int-11_ATAC_deeptools,False
1,/ceph/project/milne_group/datashare/asmith/ata...,FL10-int-11_ATAC_deeptools,False
2,/ceph/project/milne_group/datashare/asmith/ata...,FL10-int-9_ATAC_deeptools,False
3,/ceph/project/milne_group/datashare/asmith/ata...,CB9-control_ATAC_deeptools,False
4,/ceph/project/milne_group/datashare/asmith/ata...,FL10-control_ATAC_deeptools,False
...,...,...,...
707,/ceph/project/milne_group/cchahrou/data/2025-0...,RNA-SEM-1_plus,False
708,/ceph/project/milne_group/cchahrou/data/2025-0...,RNA-RCHACV-2_plus,False
709,/ceph/project/milne_group/cchahrou/data/2025-0...,RNA-SEM-1_minus,False
710,/ceph/project/milne_group/cchahrou/data/2025-0...,RNA-THP1-3_plus,False


In [39]:
bw_paths_all_df = bw_paths_all_df.drop_duplicates(subset='name')
bw_paths_all_df

,path,name
0,/ceph/project/milne_group/datashare/asmith/ata...,CB9-int-11_ATAC_deeptools
1,/ceph/project/milne_group/datashare/asmith/ata...,FL10-int-11_ATAC_deeptools
2,/ceph/project/milne_group/datashare/asmith/ata...,FL10-int-9_ATAC_deeptools
3,/ceph/project/milne_group/datashare/asmith/ata...,CB9-control_ATAC_deeptools
4,/ceph/project/milne_group/datashare/asmith/ata...,FL10-control_ATAC_deeptools
...,...,...
707,/ceph/project/milne_group/cchahrou/data/2025-0...,RNA-SEM-1_plus
708,/ceph/project/milne_group/cchahrou/data/2025-0...,RNA-RCHACV-2_plus
709,/ceph/project/milne_group/cchahrou/data/2025-0...,RNA-SEM-1_minus
710,/ceph/project/milne_group/cchahrou/data/2025-0...,RNA-THP1-3_plus


### Merge with the original dataset and remove duplicates

In [40]:
df_bw = pd.concat([bw_paths_original_df, bw_paths_datashare_df, bw_paths_all_df])
df_bw

,path,name
0,/project/milne_group/asmith/Projects/2025-07-1...,HCT116_7
1,/project/milne_group/asmith/Projects/2025-07-1...,Huh7_5
2,/project/milne_group/asmith/Projects/2025-07-1...,Huh7_3
3,/project/milne_group/asmith/Projects/2025-07-1...,786-O_2
4,/project/milne_group/asmith/Projects/2025-07-1...,786-O_3
...,...,...
707,/ceph/project/milne_group/cchahrou/data/2025-0...,RNA-SEM-1_plus
708,/ceph/project/milne_group/cchahrou/data/2025-0...,RNA-RCHACV-2_plus
709,/ceph/project/milne_group/cchahrou/data/2025-0...,RNA-SEM-1_minus
710,/ceph/project/milne_group/cchahrou/data/2025-0...,RNA-THP1-3_plus


In [42]:
df_bw['path'].to_csv('2026-05-20-dataset-paths.txt', index=False, header=False)